# Prompt 3 - Iterative Refinement Prompting

This notebook contains the implementation of the third prompting strategy used in the homework. It evaluates Qwen on the reduced `dataset_30` benchmark using an iterative refinement design, where the model generates an initial Cypher query and then attempts to verify and correct it.


In [ ]:
!pip -q install -U transformers accelerate sentencepiece gdown


In [ ]:
import os
import gdown

os.makedirs("/content/data", exist_ok=True)

LINK_DATASET_30 = "https://drive.google.com/uc?id=1IOvn9rx5-hFES6PU-3Ow8th5h-cl4z6v"
DATASET_PATH = "/content/data/dataset_30.csv"

gdown.download(LINK_DATASET_30, DATASET_PATH, quiet=False)
print("Dataset saved to:", DATASET_PATH)


In [ ]:
import json
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
import torch
from transformers import pipeline

DATASET_PATH = "/content/data/dataset_30.csv"
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
SLEEP_SECONDS = 0.1

FULL_SCHEMA = 'Node properties:\nMovie {posterEmbedding: LIST, url: STRING, runtime: INTEGER, revenue: INTEGER, budget: INTEGER, plotEmbedding: LIST, imdbRating: FLOAT, released: STRING, countries: LIST, languages: LIST, plot: STRING, imdbVotes: INTEGER, imdbId: STRING, year: INTEGER, poster: STRING, movieId: STRING, tmdbId: STRING, title: STRING}\nGenre {name: STRING}\nUser {userId: STRING, name: STRING}\nActor {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nDirector {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nPerson {url: STRING, bornIn: STRING, bio: STRING, died: DATE, born: DATE, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}\nRelationship properties:\nRATED {rating: FLOAT, timestamp: INTEGER}\nACTED_IN {role: STRING}\nDIRECTED {role: STRING}\nThe relationships:\n(:Movie)-[:IN_GENRE]->(:Genre)\n(:User)-[:RATED]->(:Movie)\n(:Actor)-[:ACTED_IN]->(:Movie)\n(:Actor)-[:DIRECTED]->(:Movie)\n(:Director)-[:DIRECTED]->(:Movie)\n(:Director)-[:ACTED_IN]->(:Movie)\n(:Person)-[:ACTED_IN]->(:Movie)\n(:Person)-[:DIRECTED]->(:Movie)'

SYSTEM_PROMPT = (
    "You are an expert Neo4j and Cypher assistant. "
    "Always answer with a single valid JSON object and nothing else. "
    "Never use SQL syntax such as GROUP BY, HAVING, JOIN, or SELECT. "
    "Use Cypher WITH for aggregation steps. "
    "Use only schema-valid labels, relationship types, and properties."
)

PIPE = None

def load_generation_pipeline(model_name: str = MODEL_NAME):
    global PIPE
    if PIPE is not None:
        return PIPE

    torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    PIPE = pipeline(
        "text-generation",
        model=model_name,
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    if PIPE.tokenizer.pad_token_id is None:
        PIPE.tokenizer.pad_token_id = PIPE.tokenizer.eos_token_id
    return PIPE

def call_model(prompt: str, max_new_tokens: int) -> str:
    pipe = load_generation_pipeline()
    outputs = pipe(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
        pad_token_id=pipe.tokenizer.pad_token_id,
    )
    generated = outputs[0]["generated_text"]
    text = generated[-1]["content"].strip() if isinstance(generated, list) else str(generated).strip()
    time.sleep(SLEEP_SECONDS)
    return text

def extract_json_block(text: str) -> Optional[Dict[str, Any]]:
    if not isinstance(text, str):
        return None

    cleaned = text.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        parsed = json.loads(cleaned)
        return parsed if isinstance(parsed, dict) else None
    except json.JSONDecodeError:
        pass

    start = cleaned.find("{")
    if start == -1:
        return None

    depth = 0
    for index in range(start, len(cleaned)):
        char = cleaned[index]
        if char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                candidate = cleaned[start : index + 1]
                try:
                    parsed = json.loads(candidate)
                    return parsed if isinstance(parsed, dict) else None
                except json.JSONDecodeError:
                    return None
    return None

def normalize_cypher(text: Any) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*,\s*", ", ", text)
    return text

def read_dataset(dataset_path: str = DATASET_PATH) -> pd.DataFrame:
    df = pd.read_csv(dataset_path)
    required = {"id", "question", "gold_cypher"}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")
    return df

def get_completed_ids(output_path: str) -> set[str]:
    path = Path(output_path)
    if not path.exists():
        return set()
    df = pd.read_csv(path)
    if "id" not in df.columns:
        return set()
    return set(df["id"].astype(str))

def append_result(output_path: str, result: Dict[str, Any]) -> None:
    row_df = pd.DataFrame([result])
    header = not Path(output_path).exists()
    row_df.to_csv(output_path, mode="a", header=header, index=False)


In [ ]:
OUTPUT_PATH = "/content/results_prompt3_qwen_dataset30.csv"
MAX_NEW_TOKENS = 180
VERIFY_MAX_NEW_TOKENS = 140
MAX_ITERS = 2

def build_generate_prompt(question: str) -> str:
    return f"""
Task:
Generate a Cypher query for the question.

Instructions:
- Use only labels, relationships, and properties present in the schema.
- Do not invent schema elements.
- Do not use SQL keywords such as GROUP BY, HAVING, JOIN, or SELECT.
- Return JSON only.

Schema:
{FULL_SCHEMA}

Question:
{question}

Return JSON:
{{
  "cypher": "generated Cypher query"
}}
""".strip()

def build_verify_prompt(question: str, cypher: str) -> str:
    return f"""
Task:
Check whether the Cypher query is valid for the schema and question.

Instructions:
- Reject SQL syntax such as GROUP BY, HAVING, JOIN, or SELECT.
- Reject invented labels, relationships, and properties.
- Reject queries that do not answer the question.
- Return JSON only.

Schema:
{FULL_SCHEMA}

Question:
{question}

Cypher:
{cypher}

Return JSON:
{{
  "valid": true,
  "error_reason": ""
}}
""".strip()

def build_correct_prompt(question: str, cypher: str, error_reason: str) -> str:
    return f"""
Task:
Fix the Cypher query using the detected error.

Instructions:
- Keep the answer in Cypher, not SQL.
- Use only schema-valid labels, relationships, and properties.
- Return JSON only.

Schema:
{FULL_SCHEMA}

Question:
{question}

Current Cypher:
{cypher}

Detected problem:
{error_reason}

Return JSON:
{{
  "cypher": "corrected Cypher query"
}}
""".strip()

def run_iterative_refinement(question: str) -> Dict[str, Any]:
    initial_raw = call_model(build_generate_prompt(question), max_new_tokens=MAX_NEW_TOKENS)
    initial_parsed = extract_json_block(initial_raw)
    current_cypher = str(initial_parsed.get("cypher", "")).strip() if initial_parsed else ""
    initial_parse_ok = initial_parsed is not None and bool(current_cypher)

    history = []
    final_valid = False

    if not initial_parse_ok:
        return {
            "initial_parse_ok": False,
            "final_valid": False,
            "iterations": 0,
            "final_cypher": "",
            "initial_raw": initial_raw,
            "history_json": json.dumps(history, ensure_ascii=True),
            "error_message": "Could not parse initial generation JSON.",
        }

    for iteration in range(1, MAX_ITERS + 1):
        verify_raw = call_model(build_verify_prompt(question, current_cypher), max_new_tokens=VERIFY_MAX_NEW_TOKENS)
        verify_parsed = extract_json_block(verify_raw)

        if not verify_parsed:
            history.append({
                "iteration": iteration,
                "cypher": current_cypher,
                "valid": False,
                "error_reason": "Verifier output could not be parsed.",
                "verify_raw": verify_raw,
                "correct_raw": "",
            })
            break

        is_valid = bool(verify_parsed.get("valid", False))
        error_reason = str(verify_parsed.get("error_reason", "")).strip()
        history_item = {
            "iteration": iteration,
            "cypher": current_cypher,
            "valid": is_valid,
            "error_reason": error_reason,
            "verify_raw": verify_raw,
            "correct_raw": "",
        }

        if is_valid:
            history.append(history_item)
            final_valid = True
            break

        correct_raw = call_model(
            build_correct_prompt(question, current_cypher, error_reason),
            max_new_tokens=MAX_NEW_TOKENS,
        )
        correct_parsed = extract_json_block(correct_raw)
        history_item["correct_raw"] = correct_raw
        history.append(history_item)

        if not correct_parsed:
            break

        updated_cypher = str(correct_parsed.get("cypher", "")).strip()
        if not updated_cypher or updated_cypher == current_cypher:
            break
        current_cypher = updated_cypher

    return {
        "initial_parse_ok": initial_parse_ok,
        "final_valid": final_valid,
        "iterations": len(history),
        "final_cypher": current_cypher,
        "initial_raw": initial_raw,
        "history_json": json.dumps(history, ensure_ascii=True),
        "error_message": "",
    }

def run_prompt3(resume: bool = True) -> pd.DataFrame:
    df = read_dataset()
    completed_ids = get_completed_ids(OUTPUT_PATH) if resume else set()

    for _, row in df.iterrows():
        if str(row["id"]) in completed_ids:
            continue

        payload = run_iterative_refinement(row["question"])
        result = {
            "id": row["id"],
            "difficulty": row.get("difficulty", ""),
            "source_type": row.get("source_type", ""),
            "source_row": row.get("source_row", ""),
            "question": row["question"],
            "gold_cypher": row.get("gold_cypher", ""),
            "prompt_name": "prompt_3_iterative_refinement_qwen",
            "model_name": MODEL_NAME,
            "initial_parse_ok": payload["initial_parse_ok"],
            "final_valid": payload["final_valid"],
            "iterations": payload["iterations"],
            "predicted_cypher": payload["final_cypher"],
            "exact_match": normalize_cypher(payload["final_cypher"]) == normalize_cypher(row["gold_cypher"]),
            "initial_raw": payload["initial_raw"],
            "history_json": payload["history_json"],
            "error_message": payload["error_message"],
        }
        append_result(OUTPUT_PATH, result)
        print(f"[{row['id']}] final_valid={result['final_valid']} iterations={result['iterations']} exact_match={result['exact_match']}")

    return pd.read_csv(OUTPUT_PATH)

df_results = run_prompt3(resume=True)
df_results.head()
